In [28]:
import requests

GROBID_URL = "http://localhost:8070"

with open("../data/2302.13971.pdf", "rb") as f:
    resp = requests.post(
        f"{GROBID_URL}/api/processFulltextDocument",
        files={"input": ("2302.13971.pdf", f, "application/pdf")},
        data={"generateIDs": "1", "consolidateHeader": "1"},
        timeout=120,
    )

print(f"Status: {resp.status_code}")
xml_out = resp.text

Status: 200


In [29]:
from xml.etree import ElementTree as ET

import pandas as pd

NS = "{http://www.tei-c.org/ns/1.0}"
root = ET.fromstring(xml_out)


def parse_ref(bibl):
    authors = []
    for a in bibl.findall(f".//{NS}author/{NS}persName"):
        first = a.findtext(f"{NS}forename", default="")
        last = a.findtext(f"{NS}surname", default="")
        authors.append(f"{first} {last}".strip())

    title = ""
    for parent_tag in [f"{NS}analytic", f"{NS}monogr"]:
        el = bibl.find(f"{parent_tag}/{NS}title")
        if el is not None and el.text:
            title = el.text.strip()
            break
    if not title:
        note = bibl.find(f"{NS}note[@type='report_type']")
        if note is not None and note.text:
            title = (
                note.text.strip()
                .removesuffix(". arXiv preprint")
                .removesuffix("arXiv preprint")
            )

    venue = ""
    monogr_title = bibl.find(f"{NS}monogr/{NS}title")
    if (
        monogr_title is not None
        and monogr_title.text
        and monogr_title.text.strip() != title
    ):
        venue = monogr_title.text.strip()

    year = None
    date_el = bibl.find(f".//{NS}date[@when]")
    if date_el is not None:
        try:
            year = int(date_el.get("when", "")[:4])
        except ValueError, TypeError:
            pass

    doi = None
    doi_el = bibl.find(f".//{NS}idno[@type='DOI']")
    if doi_el is not None and doi_el.text:
        doi = doi_el.text.strip()

    return {
        "ref_id": bibl.get("{http://www.w3.org/XML/1998/namespace}id", ""),
        "authors": "; ".join(authors),
        "title": title,
        "venue": venue,
        "year": year,
        "doi": doi,
    }


refs = [parse_ref(b) for b in root.findall(f".//{NS}listBibl/{NS}biblStruct")]
print(f"{len(refs)} references extracted")
pd.DataFrame(refs)

54 references extracted


,ref_id,authors,title,venue,year,doi
0,b0,Jacob Austin; Augustus Odena; Maxwell Nye; Maa...,,,2021.0,None
1,b1,Frederick Lalit R Bahl; Robert Jelinek; Mercer,A maximum likelihood approach to continuous sp...,IEEE transactions on pattern analysis and mach...,1983.0,None
2,b2,Yoshua Bengio; Réjean Ducharme; Pascal Vincent,A neural probabilistic language model. Advance...,,2000.0,None
3,b3,Yonatan Bisk; Rowan Zellers; Jianfeng Gao; Yej...,Piqa: Reasoning about physical commonsense in ...,Proceedings of the AAAI conference on artifici...,2020.0,None
4,b4,Sid Black; Stella Biderman; Eric Hallahan; Que...,Gpt-neox-20b: An open-source autoregressive la...,,2022.0,None
5,b5,Thorsten Brants; C Ashok; Peng Popat; Franz Xu...,Large language models in machine translation,Proceedings of the 2007 Joint Conference on Em...,2007.0,None
6,b6,John Peter F Brown; Stephen Cocke; Della Pietr...,A statistical approach to machine translation,Computational linguistics,1990.0,None
7,b7,Rafal Jozefowicz; Oriol Vinyals; Mike Schuster...,Exploring the limits of language modeling,,2016.0,None
8,b8,Jared Kaplan; Sam Mccandlish; Tom Henighan; To...,Scaling laws for neural language models,,2020.0,None
9,b9,Slava Katz,Estimation of probabilities from sparse data f...,"IEEE transactions on acoustics, speech, and si...",1987.0,None
